# LightEval CSD decoding sweep plots

This notebook scans all result JSONL files under `benchmark/csd/runs/lighteval_csd_decoding_comparison_sweep`, then generates clear CSD sweep figures for every run that has results.

**Baseline definition:** baseline means vanilla EAGLE / naive speculative decoding. All speedups and deltas in these plots are computed relative to that baseline.

**Filtering:** CSD `prob_ratio=0.01` rows are filtered out before plotting. Accuracy always uses the JSONL top-level `accuracy` field.

**Figure style:** each figure title includes the model name at the top; bottom source/footer text is omitted.

Each run writes figures to its own `figures/` directory next to `results/`.

In [ ]:
from pathlib import Path
import importlib
import sys

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / 'benchmark/csd/runs/lighteval_csd_decoding_comparison_sweep').exists():
            return path
    fallback = Path('/root/sglang')
    if fallback.exists():
        return fallback
    raise FileNotFoundError('Could not find the sglang repository root')

REPO_ROOT = find_repo_root()
SWEEP_DIR = REPO_ROOT / 'benchmark/csd/runs/lighteval_csd_decoding_comparison_sweep'
EVAL_DIR = REPO_ROOT / 'benchmark/csd/eval'

if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))

import plot_lighteval_csd_decoding_sweep as plot_sweep
plot_sweep = importlib.reload(plot_sweep)

print('Repo root:', REPO_ROOT)
print('Sweep dir:', SWEEP_DIR)
print('Baseline:', plot_sweep.BASELINE_LABEL)
print('Filtered CSD prob ratios:', sorted(plot_sweep.EXCLUDED_CSD_PROB_RATIOS))

In [ ]:
RESULT_FILES = sorted(SWEEP_DIR.glob('*/results/csd_decoding_comparison_sweep.jsonl'))
if not RESULT_FILES:
    raise FileNotFoundError(f'No result JSONL files found under {SWEEP_DIR}')

for path in RESULT_FILES:
    line_count = sum(1 for line in path.open() if line.strip())
    plotted_rows = len(plot_sweep.load_rows(path))
    print(f'{path.relative_to(REPO_ROOT)}: {line_count} raw rows, {plotted_rows} plotted rows')

In [ ]:
OUTPUT_DIRS = []
for jsonl_path in RESULT_FILES:
    print(f'Plotting {jsonl_path.relative_to(REPO_ROOT)}')
    out_dir = plot_sweep.plot_all_for_result_file(jsonl_path, repo_root=REPO_ROOT)
    OUTPUT_DIRS.append(out_dir)
    print(f'  wrote {out_dir.relative_to(REPO_ROOT)}')

print(f'Finished {len(OUTPUT_DIRS)} runs.')

In [ ]:
from IPython.display import Image, Markdown, display

for out_dir in OUTPUT_DIRS:
    display(Markdown(f'## `{out_dir.parent.name}`'))
    display(Image(filename=str(out_dir / 'csd_sweep_overview_best_throughput.png')))
    for task in plot_sweep.TASK_ORDER:
        dashboard = out_dir / f'csd_sweep_{task.lower()}_baseline_dashboard.png'
        if dashboard.exists():
            display(Image(filename=str(dashboard)))
    print('All files:')
    for path in sorted(out_dir.iterdir()):
        print(' ', path.relative_to(REPO_ROOT))